# 12 — In-group / Out-group dataset (Table 6)

Tag every rule of the final dataset as **in-group (privileged)** or **out-group (deprived)** for each criterion, following the manual assignment in Table 6 of the paper.

Output: `data/clean/final/rules_in_out_groups.tsv` — one row per (rule × matched criterion × matched group).

## 1. Setup

In [1]:
import ast
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 80)

RULES_PATH = Path("../data/clean/final/rules_final_dataset_130works_april_2026.tsv")
OUT_PATH = Path("../data/clean/final/rules_in_out_groups.tsv")
SUMMARY_PATH = Path("../data/clean/final/in_out_groups_summary.tsv")

## 2. Load rules

In [2]:
rules = pd.read_csv(RULES_PATH, sep="\t")
print(f"rules loaded: {len(rules):,} rows, {rules.shape[1]} cols")
assert rules["rule_uid"].is_unique, "rule_uid should be unique"
rules[["rule_uid", "criteria", "group", "group_meta"]].head()

rules loaded: 1,011 rows, 63 cols


,rule_uid,criteria,group,group_meta
0,tlg0007.tlg118.perseus-eng3::0,['Ethnicity'],Greeks,Greeks
1,tlg0007.tlg118.perseus-eng3::1,['Citizenship'],Citizens of Greek city-states,Citizens;Greeks
2,tlg0007.tlg118.perseus-eng3::2,['Nobility'],Nobles,Nobles
3,tlg0007.tlg118.perseus-eng3::3,['Citizenship'],Citizens,Citizens
4,tlg0007.tlg118.perseus-eng3::5,['Citizenship'],Citizens,Citizens


## 3. Table 6 — manual in-group / out-group mapping

Each criterion has a privileged (in-group) and a deprived (out-group) side. `group_meta` values are matched against these sets.

In [3]:
TABLE_6 = {
    "Gender":              {"in": ["Men"],          "out": ["Women"]},
    "Age":                 {"in": ["Elders"],       "out": ["Minors"]},
    "Ethnicity":           {"in": ["Greeks"],       "out": ["Syrians", "Persians", "Scythians"]},
    "Citizenship":         {"in": ["Citizens"],     "out": ["Foreigners", "Slaves", "Women"]},
    "Freedom":             {"in": ["Citizens"],     "out": ["Slaves"]},
    "Occupation":          {"in": ["Artisans"],     "out": ["Citizens"]},
    "Lineage":             {"in": ["Heirs"],        "out": ["Citizens"]},
    "Nobility":            {"in": ["Nobles"],       "out": ["The multitude", "Citizens"]},
    "Wealth / Properties": {"in": ["The wealthy"],  "out": ["The poor"]},
    "Education":           {"in": ["The educated"], "out": ["Citizens"]},
    "Religion":            {"in": ["Priests"],      "out": ["Citizens"]},
}

table6_df = pd.DataFrame(
    [
        {"criterion": c, "in_group": ", ".join(v["in"]), "out_group": ", ".join(v["out"])}
        for c, v in TABLE_6.items()
    ]
)
table6_df

,criterion,in_group,out_group
0,Gender,Men,Women
1,Age,Elders,Minors
2,Ethnicity,Greeks,"Syrians, Persians, Scythians"
3,Citizenship,Citizens,"Foreigners, Slaves, Women"
4,Freedom,Citizens,Slaves
5,Occupation,Artisans,Citizens
6,Lineage,Heirs,Citizens
7,Nobility,Nobles,"The multitude, Citizens"
8,Wealth / Properties,The wealthy,The poor
9,Education,The educated,Citizens


## 4. Parse `criteria` and `group_meta`

`criteria` is stored as a stringified list. `group_meta` is a `;`-separated string (a rule can target several meta-groups at once, e.g. `Men;Heirs`).

In [4]:
def parse_list_field(s):
    if not isinstance(s, str) or not s.strip():
        return []
    try:
        v = ast.literal_eval(s)
        return [x.strip() for x in v] if isinstance(v, list) else [str(v).strip()]
    except (ValueError, SyntaxError):
        return [s.strip()]


def parse_group_meta(s):
    if not isinstance(s, str):
        return []
    return [g.strip() for g in s.split(";") if g.strip()]


rules["criteria_list"] = rules["criteria"].apply(parse_list_field)
rules["group_meta_list"] = rules["group_meta"].apply(parse_group_meta)
rules[["criteria", "criteria_list", "group_meta", "group_meta_list"]].head()

,criteria,criteria_list,group_meta,group_meta_list
0,['Ethnicity'],[Ethnicity],Greeks,[Greeks]
1,['Citizenship'],[Citizenship],Citizens;Greeks,"[Citizens, Greeks]"
2,['Nobility'],[Nobility],Nobles,[Nobles]
3,['Citizenship'],[Citizenship],Citizens,[Citizens]
4,['Citizenship'],[Citizenship],Citizens,[Citizens]


## 5. Tag each rule as in-group / out-group

In [5]:
KEEP_COLS = [
    "rule_uid", "file_id", "perseus_author", "perseus_title",
    "year", "period", "rule", "group", "group_meta", "criteria",
    "resource", "resource_meta", "directionality", "verbatim",
]

records = []
for _, row in rules.iterrows():
    for criterion in row["criteria_list"]:
        spec = TABLE_6.get(criterion)
        if spec is None:
            continue
        for meta in row["group_meta_list"]:
            if meta in spec["in"]:
                side = "in-group (privileged)"
            elif meta in spec["out"]:
                side = "out-group (deprived)"
            else:
                continue
            rec = {c: row[c] for c in KEEP_COLS}
            rec["criterion"] = criterion
            rec["group_matched"] = meta
            rec["group_side"] = side
            records.append(rec)

in_out = pd.DataFrame(records)
print(f"tagged pairings: {len(in_out):,}  (rules × matched criterion × matched group)")
print(f"unique rules covered: {in_out['rule_uid'].nunique():,} / {len(rules):,}")
in_out.head()

tagged pairings: 868  (rules × matched criterion × matched group)
unique rules covered: 786 / 1,011


,rule_uid,file_id,perseus_author,perseus_title,year,period,rule,group,group_meta,criteria,resource,resource_meta,directionality,verbatim,criterion,group_matched,group_side
0,tlg0007.tlg118.perseus-eng3::0,tlg0007.tlg118.perseus-eng3,Plutarch,Precepts of Statecraft,75,Hellenistic & Early Roman (165 BCE – 105 CE),Greek exclusion from large-scale statecraft,Greeks,Greeks,['Ethnicity'],Access to high-level imperial governance,Political power,LESS,['The exercise of statecraft on a large scale was virtually limited to Roman...,Ethnicity,Greeks,in-group (privileged)
1,tlg0007.tlg118.perseus-eng3::1,tlg0007.tlg118.perseus-eng3,Plutarch,Precepts of Statecraft,75,Hellenistic & Early Roman (165 BCE – 105 CE),City-state local self-government,Citizens of Greek city-states,Citizens;Greeks,['Citizenship'],Right to local self-government,Political power,MORE,"['The ancient Greek city-states retained, however, their local self-governme...",Citizenship,Citizens,in-group (privileged)
2,tlg0007.tlg118.perseus-eng3::2,tlg0007.tlg118.perseus-eng3,Plutarch,Precepts of Statecraft,75,Hellenistic & Early Roman (165 BCE – 105 CE),Noble access to magistracies,Nobles,Nobles,['Nobility'],Eligibility for the archonship,Eligibility for public office,MORE,"['Therefore, seeing that the desire has been aroused in you a Speaker of spe...",Nobility,Nobles,in-group (privileged)
3,tlg0007.tlg118.perseus-eng3::3,tlg0007.tlg118.perseus-eng3,Plutarch,Precepts of Statecraft,75,Hellenistic & Early Roman (165 BCE – 105 CE),Citizen assembly voting,Citizens,Citizens,['Citizenship'],Right to vote in the Ekklesia,Right to vote in the Ekklesia,MORE,"['Take care, Pericles; you are ruling free men, you are ruling Greeks, Athen...",Citizenship,Citizens,in-group (privileged)
4,tlg0007.tlg118.perseus-eng3::5,tlg0007.tlg118.perseus-eng3,Plutarch,Precepts of Statecraft,75,Hellenistic & Early Roman (165 BCE – 105 CE),Proconsular annulment of local power,Citizens,Citizens,['Citizenship'],Legislative finality,Political power,LESS,"['For what dominion, what glory is there for those who are victorious? What ...",Citizenship,Citizens,in-group (privileged)


## 6. Counts per criterion (in-group vs out-group)

Reproduces Table 6 in numbers derived from the final dataset.

In [6]:
summary = (
    in_out.groupby(["criterion", "group_side", "group_matched"])
    .size()
    .reset_index(name="n")
    .sort_values(["criterion", "group_side", "n"], ascending=[True, True, False])
)
summary

,criterion,group_side,group_matched,n
0,Age,in-group (privileged),Elders,8
1,Age,out-group (deprived),Minors,34
2,Citizenship,in-group (privileged),Citizens,266
3,Citizenship,out-group (deprived),Foreigners,74
5,Citizenship,out-group (deprived),Women,10
4,Citizenship,out-group (deprived),Slaves,6
6,Education,in-group (privileged),The educated,2
7,Education,out-group (deprived),Citizens,3
8,Ethnicity,in-group (privileged),Greeks,2
10,Ethnicity,out-group (deprived),Scythians,3


In [7]:
pivot = (
    in_out.groupby(["criterion", "group_side"]).size().unstack(fill_value=0)
)
pivot = pivot.reindex(list(TABLE_6.keys()))
pivot["total"] = pivot.sum(axis=1)
pivot

group_side,in-group (privileged),out-group (deprived),total
criterion,,,
Gender,15,75,90
Age,8,34,42
Ethnicity,2,4,6
Citizenship,266,90,356
Freedom,29,77,106
Occupation,21,10,31
Lineage,22,14,36
Nobility,32,1,33
Wealth / Properties,100,51,151


## 7. Save

In [8]:
in_out.to_csv(OUT_PATH, sep="\t", index=False)
summary.to_csv(SUMMARY_PATH, sep="\t", index=False)
print(f"wrote {OUT_PATH}  ({len(in_out):,} rows)")
print(f"wrote {SUMMARY_PATH}  ({len(summary):,} rows)")

wrote ../data/clean/final/rules_in_out_groups.tsv  (868 rows)
wrote ../data/clean/final/in_out_groups_summary.tsv  (25 rows)
